# Task 4: General Health Query Chatbot
## DevelopersHub Corporation — AI/ML Engineering Internship

---

### Problem Statement
Build a conversational chatbot that answers general health-related questions using a Large Language Model (LLM).  
The chatbot must be:
- **Friendly and informative** — using prompt engineering to shape tone and behaviour
- **Safe** — includes filters to block harmful or dangerous medical advice
- **Conversational** — maintains multi-turn dialogue history

### Goal
Demonstrate prompt engineering, LLM API integration, safety handling, and building a simple conversational agent.

### Tools & Model
- **Model:** `claude-sonnet-4-20250514` via Anthropic API (equivalent to GPT-3.5 tier as specified in the task)
- **Safety Filters:** Rule-based keyword detection + LLM-level prompt constraints
- **Interface:** Interactive Python (works in Jupyter or CLI)

---

## 1. Setup & Dependencies

In [ ]:
# Install required libraries
# Run this cell once
!pip install anthropic --quiet

# If using OpenAI instead, run:
# !pip install openai --quiet

In [ ]:
import os
import re
import anthropic

print("✅ Libraries loaded successfully.")

## 2. API Key Configuration

> **Security Note:** Never hardcode API keys. Always use environment variables or `.env` files.
> Store your key in an environment variable: `export ANTHROPIC_API_KEY="your-key-here"`

In [ ]:
# Load the API key from environment (never hardcode it!)
API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not API_KEY:
    raise ValueError(
        "❌ API key not found! Please set ANTHROPIC_API_KEY as an environment variable.\n"
        "  In terminal: export ANTHROPIC_API_KEY='your-key-here'\n"
        "  In Jupyter:  %env ANTHROPIC_API_KEY=your-key-here"
    )

# Initialize the client
client = anthropic.Anthropic(api_key=API_KEY)
print("✅ Anthropic client initialized.")

## 3. Prompt Engineering — System Prompt Design

This is the **core of Task 4**. The system prompt shapes the entire behaviour of the chatbot:
- Defines its persona (friendly health assistant)
- Sets boundaries (no diagnoses, no prescriptions)
- Enforces safety (always recommend seeing a doctor for serious symptoms)

In [ ]:
# =============================================================================
# SYSTEM PROMPT — Defines the chatbot's identity, tone, and safety rules
# =============================================================================

SYSTEM_PROMPT = """
You are HealthBot, a friendly and knowledgeable general health information assistant.

## Your Role
- Answer general health questions clearly, warmly, and in plain language.
- Provide educational health information based on established medical knowledge.
- Help users understand common symptoms, conditions, medications, and wellness tips.

## Your Tone
- Be warm, empathetic, and reassuring — like a knowledgeable friend.
- Use simple language. Avoid excessive medical jargon.
- Structure answers with short paragraphs or bullet points for clarity.

## STRICT Safety Rules — You MUST follow these at all times
1. NEVER diagnose a user with a specific medical condition.
2. NEVER prescribe or recommend specific medications, dosages, or treatments.
3. NEVER advise users to stop taking prescribed medications.
4. If a user describes a SERIOUS or EMERGENCY symptom (chest pain, difficulty breathing,
   severe bleeding, stroke symptoms, suicidal thoughts), IMMEDIATELY direct them to
   call emergency services (911 or local equivalent) and do NOT provide other advice.
5. Always end responses about symptoms or conditions with a recommendation to consult
   a qualified healthcare professional.
6. If unsure about any health information, say so clearly and recommend a doctor.

## What You Can Help With
- Explaining what common conditions or terms mean
- General wellness advice (sleep, nutrition, hydration, exercise)
- How common over-the-counter medications generally work (not specific dosing)
- Answering "what causes X?" or "what is X?" questions
- First-aid basics for minor injuries

## Disclaimer
Always remind users that your information is educational only and does not replace
professional medical advice, diagnosis, or treatment.
""".strip()

## 4. Safety Filter — Pre-LLM Keyword Detection

A **two-layer safety approach** is used:
1. **Layer 1 (Pre-filter):** Rule-based keyword detection before the query even reaches the LLM
2. **Layer 2 (LLM-level):** The system prompt instructs the model itself to handle edge cases

This is best practice in production health chatbots.

In [ ]:
# =============================================================================
# SAFETY FILTER — Rule-based pre-screening before calling the LLM
# =============================================================================

# Emergency keywords — triggers immediate emergency response
EMERGENCY_KEYWORDS = [
    r"\bchest pain\b", r"\bcan'?t breathe\b", r"\bcan not breathe\b",
    r"\bdifficulty breathing\b", r"\bshortness of breath\b",
    r"\bheart attack\b", r"\bstroke\b", r"\blosing consciousness\b",
    r"\bpassing out\b", r"\bsevere bleeding\b", r"\bsuicid\b",
    r"\bkill myself\b", r"\bend my life\b", r"\boverdos\b",
    r"\bseizure\b", r"\bunconscious\b",
]

# Harmful request keywords — blocks dangerous advice requests
HARMFUL_KEYWORDS = [
    r"\bhow to make\b.{0,30}\b(poison|drug|toxin)\b",
    r"\blethal dose\b", r"\bhow much .{0,20} to die\b",
    r"\bhow to harm\b", r"\bhow to hurt\b",
]

EMERGENCY_RESPONSE = (
    "🚨 **This sounds like a medical emergency.**\n\n"
    "Please **call emergency services immediately** (911 in the US, 115 in Pakistan, "
    "or your local emergency number).\n\n"
    "Do not wait — emergency situations require immediate professional care. "
    "If you're with someone who is unwell, stay with them and call for help now."
)

HARMFUL_RESPONSE = (
    "⚠️ I'm not able to help with that request. "
    "If you're going through a difficult time, please reach out to a trusted person or "
    "a mental health professional. You can also call a crisis helpline for support."
)


def check_safety(user_input: str) -> tuple[bool, str]:
    """
    Pre-screens user input before sending to the LLM.

    Returns:
        (is_blocked: bool, response: str)
        If is_blocked is True, return the response directly without calling LLM.
    """
    lowered = user_input.lower()

    # Check for emergency keywords
    for pattern in EMERGENCY_KEYWORDS:
        if re.search(pattern, lowered):
            return True, EMERGENCY_RESPONSE

    # Check for harmful request patterns
    for pattern in HARMFUL_KEYWORDS:
        if re.search(pattern, lowered):
            return True, HARMFUL_RESPONSE

    return False, ""


# Quick test of safety filter
test_cases = [
    ("I have chest pain and can't breathe", True),
    ("What causes a sore throat?", False),
    ("Is paracetamol safe for children?", False),
]

print("Safety Filter Tests:")
print("-" * 50)
for query, expected_block in test_cases:
    blocked, _ = check_safety(query)
    status = "✅ PASS" if blocked == expected_block else "❌ FAIL"
    print(f"{status} | Blocked={blocked} | Query: '{query}'")

## 5. Chatbot Core — Multi-Turn Conversation Engine

In [ ]:
# =============================================================================
# CHATBOT ENGINE — Handles conversation history and LLM calls
# =============================================================================

class HealthChatbot:
    """
    A multi-turn health information chatbot powered by an LLM.
    
    Features:
    - Maintains conversation history for context-aware responses
    - Two-layer safety filtering (pre-LLM + LLM-level)
    - Friendly, empathetic tone via prompt engineering
    """

    def __init__(self, client, system_prompt: str, model: str = "claude-sonnet-4-20250514"):
        self.client = client
        self.system_prompt = system_prompt
        self.model = model
        self.conversation_history = []  # Stores full dialogue for context

    def chat(self, user_input: str) -> str:
        """
        Send a message and get a response. Handles safety filtering and history.
        
        Args:
            user_input: The user's message
        
        Returns:
            The chatbot's response as a string
        """
        # --- Layer 1: Pre-LLM Safety Check ---
        is_blocked, safety_response = check_safety(user_input)
        if is_blocked:
            return safety_response

        # Add user message to conversation history
        self.conversation_history.append({
            "role": "user",
            "content": user_input
        })

        # --- Layer 2: Call LLM with full conversation context ---
        try:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                system=self.system_prompt,          # System prompt sets the persona
                messages=self.conversation_history,  # Full history for multi-turn context
            )

            # Extract the text response
            assistant_message = response.content[0].text

            # Add assistant response to history
            self.conversation_history.append({
                "role": "assistant",
                "content": assistant_message
            })

            return assistant_message

        except Exception as e:
            # Remove the user message if API call failed
            self.conversation_history.pop()
            return f"⚠️ Error communicating with the AI service: {str(e)}"

    def reset(self):
        """Clear conversation history to start a fresh session."""
        self.conversation_history = []
        print("🔄 Conversation history cleared. Starting fresh!")

    def show_history(self):
        """Display the full conversation history."""
        if not self.conversation_history:
            print("No conversation history yet.")
            return
        for i, msg in enumerate(self.conversation_history):
            role = "👤 You" if msg["role"] == "user" else "🤖 HealthBot"
            print(f"\n[{i+1}] {role}:")
            print(msg["content"])


# Instantiate the chatbot
bot = HealthChatbot(client=client, system_prompt=SYSTEM_PROMPT)
print("✅ HealthBot initialized and ready!")

## 6. Test Runs — Example Queries

These demonstrate the chatbot's capabilities across different query types:
- General symptom question
- Medication safety question  
- Wellness / lifestyle question
- Emergency detection (safety filter)
- Multi-turn follow-up (tests conversation memory)

In [ ]:
# =============================================================================
# TEST 1: General Symptom Question
# =============================================================================
print("=" * 60)
print("TEST 1: General Symptom Question")
print("=" * 60)

query1 = "What causes a sore throat?"
print(f"\n👤 User: {query1}")
print(f"\n🤖 HealthBot:\n")

response1 = bot.chat(query1)
print(response1)

In [ ]:
# =============================================================================
# TEST 2: Medication Safety Question
# =============================================================================
print("=" * 60)
print("TEST 2: Medication Safety Question")
print("=" * 60)

query2 = "Is paracetamol safe for children?"
print(f"\n👤 User: {query2}")
print(f"\n🤖 HealthBot:\n")

response2 = bot.chat(query2)
print(response2)

In [ ]:
# =============================================================================
# TEST 3: Wellness / Lifestyle Tip
# =============================================================================
print("=" * 60)
print("TEST 3: Wellness / Lifestyle Tip")
print("=" * 60)

query3 = "How much water should I drink daily? I keep forgetting to stay hydrated."
print(f"\n👤 User: {query3}")
print(f"\n🤖 HealthBot:\n")

response3 = bot.chat(query3)
print(response3)

In [ ]:
# =============================================================================
# TEST 4: Multi-Turn Follow-Up (tests conversation memory)
# =============================================================================
print("=" * 60)
print("TEST 4: Multi-Turn Follow-Up")
print("=" * 60)

# Reset history for a clean multi-turn test
bot.reset()

query4a = "I've been feeling really tired lately and have a mild headache."
print(f"\n👤 User: {query4a}")
print(f"\n🤖 HealthBot:\n")
print(bot.chat(query4a))

print("\n" + "-" * 40)

query4b = "Could it be related to dehydration? I don't drink much water."
print(f"\n👤 User: {query4b}")
print(f"\n🤖 HealthBot:\n")
print(bot.chat(query4b))

print("\n" + "-" * 40)

query4c = "What else could cause these symptoms?"
print(f"\n👤 User: {query4c}")
print(f"\n🤖 HealthBot:\n")
print(bot.chat(query4c))

In [ ]:
# =============================================================================
# TEST 5: Emergency Detection (Safety Filter Layer 1)
# =============================================================================
print("=" * 60)
print("TEST 5: Emergency Detection (Safety Filter)")
print("=" * 60)

# This should be intercepted BEFORE reaching the LLM
emergency_query = "I'm having severe chest pain and I can't breathe properly!"
print(f"\n👤 User: {emergency_query}")
print(f"\n🤖 HealthBot:\n")
print(bot.chat(emergency_query))

## 7. Interactive Mode (Run in Terminal / Jupyter)

Uncomment and run the cell below to have a live conversation with HealthBot.

Commands:
- Type your question and press Enter
- Type `reset` to clear history
- Type `history` to see the conversation
- Type `quit` or `exit` to stop

In [ ]:
# =============================================================================
# INTERACTIVE CHATBOT SESSION
# Uncomment the code below and run this cell to chat interactively
# =============================================================================

# bot.reset()  # Start fresh

# print("="*60)
# print("🏥 HealthBot — Your General Health Information Assistant")
# print("="*60)
# print("Ask any general health question. Type 'quit' to exit.")
# print("Disclaimer: This bot provides educational info only, not medical advice.\n")

# while True:
#     user_input = input("👤 You: ").strip()
#     if not user_input:
#         continue
#     if user_input.lower() in ["quit", "exit", "bye"]:
#         print("🤖 HealthBot: Take care and stay healthy! 👋")
#         break
#     elif user_input.lower() == "reset":
#         bot.reset()
#     elif user_input.lower() == "history":
#         bot.show_history()
#     else:
#         response = bot.chat(user_input)
#         print(f"\n🤖 HealthBot: {response}\n")

## 8. Results & Key Insights

### What Was Built
A multi-turn health information chatbot with:
- **Prompt Engineering:** A detailed system prompt defines the chatbot's persona, tone, scope, and hard safety rules
- **Two-Layer Safety:** Rule-based pre-filter catches emergencies instantly; LLM-level instructions handle nuanced cases
- **Conversation Memory:** Full message history passed to the LLM on every turn for coherent multi-turn dialogue
- **Graceful Error Handling:** API failures don't crash the session

### Prompt Engineering Techniques Used
| Technique | Implementation |
|---|---|
| **Role assignment** | "You are HealthBot, a friendly health assistant" |
| **Tone definition** | "Warm, empathetic, like a knowledgeable friend" |
| **Hard constraints** | Numbered safety rules with NEVER/ALWAYS keywords |
| **Scope definition** | Clear list of what the bot can and cannot help with |
| **Output format guidance** | "Use short paragraphs or bullet points" |
| **Disclaimer injection** | Always remind users this is educational |

### Safety Design
- **Layer 1 (Pre-LLM):** 15+ regex patterns for emergency and harmful keywords — zero LLM cost, instant response
- **Layer 2 (LLM-level):** System prompt rules prevent the model from diagnosing, prescribing, or giving dangerous advice
- **Fail-safe:** Emergency queries always route to emergency services guidance

### Limitations & Next Steps
- The bot relies on general LLM knowledge — it should not replace real medical consultation
- Could be enhanced with a RAG (Retrieval-Augmented Generation) pipeline using verified medical databases (PubMed, WHO)
- A Streamlit web interface could make it more user-friendly for non-technical users
- Could add conversation logging for quality monitoring in a production setting